# Lab 9 - building a leak-free pipeline

**Session 9.** Mixed numeric and categorical columns, missing values, and a
`ColumnTransformer`. Then introduce a leak on purpose and measure what it buys you.

## 1. A messy tabular dataset

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 800
df = pd.DataFrame({
    "area_sqm": rng.normal(70, 20, n).round(1),
    "dist_metro_km": rng.gamma(2, 0.6, n).round(2),
    "built_year": rng.integers(1900, 2024, n),
    "district": rng.choice(["Nordstadt", "Altmarkt", "Weststadt", "Suedhafen"], n,
                           p=[0.3, 0.25, 0.25, 0.2]),
    "heating": rng.choice(["gas", "district", "heat_pump"], n, p=[0.5, 0.35, 0.15]),
})
# A target with a real but modest signal, plus noise.
# The truth is NOT linear in the raw columns: what matters is space relative to
# centrality, and age enters with a bend. That is why section 4's features can pay.
ratio = df.area_sqm / (df.dist_metro_km + 0.1)
logit = (0.9 * np.log(ratio) - 0.004 * (2026 - df.built_year)
         + 0.5 * (df.dist_metro_km < 0.8) + rng.normal(0, 0.8, n))
df["premium"] = (logit > logit.mean()).astype(int)      # 1 = above-median rent premium

# Missing values, as in any real extract.
for col, rate in [("area_sqm", 0.08), ("built_year", 0.05), ("heating", 0.10)]:
    df.loc[rng.random(n) < rate, col] = np.nan

print(df.head())
print(f"\nbase rate {df.premium.mean():.3f}")
print(df.isna().sum().rename("missing").to_frame().T)

## 2. The pipeline

Every fitted transformation lives inside it. `add_indicator=True` keeps "this was blank" as
a feature - non-response is a behaviour.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

y = df.pop("premium")
num_cols = ["area_sqm", "dist_metro_km", "built_year"]
cat_cols = ["district", "heating"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
print(model)

## 3. Cross-validate it

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=0)
clean = cross_val_score(model, df, y, cv=cv, scoring="roc_auc")
print(f"clean pipeline: AUC {clean.mean():.3f} (sd {clean.std():.3f})")

boosted = Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(random_state=0))])
gb = cross_val_score(boosted, df, y, cv=cv, scoring="roc_auc")
print(f"same pipeline, boosted trees: AUC {gb.mean():.3f} (sd {gb.std():.3f})")

## 4. Feature engineering that earns its place

Ratios and decomposed dates, added deliberately - not forty columns because they existed.

In [ ]:
from sklearn.preprocessing import FunctionTransformer


def add_features(frame):
    out = frame.copy()
    out["age_years"] = 2026 - out["built_year"]
    out["central"] = (out["dist_metro_km"] < 0.8).astype(float)
    out["area_per_km"] = out["area_sqm"] / (out["dist_metro_km"] + 0.1)
    return out


engineered = Pipeline([
    ("fe", FunctionTransformer(add_features)),
    ("pre", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]),
         num_cols + ["age_years", "central", "area_per_km"]),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])
eng = cross_val_score(engineered, df, y, cv=cv, scoring="roc_auc")
print(f"engineered features: AUC {eng.mean():.3f} (sd {eng.std():.3f})")
print(f"change versus the plain pipeline: {eng.mean() - clean.mean():+.3f} AUC")
print("Three columns, chosen because the domain suggested them - not forty because they")
print("were available. Compare that gain with what tuning the classifier would buy.")

## 5. Now leak on purpose

Two leaks, both of which look like ordinary tidying.

In [ ]:
# Leak A: impute and scale over ALL rows before cross-validating.
leaked_A = df.copy()
leaked_A[num_cols] = leaked_A[num_cols].fillna(leaked_A[num_cols].median())
leaked_A[cat_cols] = leaked_A[cat_cols].fillna("Missing")
a = cross_val_score(
    Pipeline([("pre", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])),
              ("clf", LogisticRegression(max_iter=1000))]),
    leaked_A, y, cv=cv, scoring="roc_auc")

# Leak B: a column recorded AFTER the outcome is known.
leaked_B = df.copy()
rng2 = np.random.default_rng(7)
leaked_B["surveyor_note"] = np.where(rng2.random(len(y)) < 0.85, y, 1 - y)
b = cross_val_score(
    Pipeline([("pre", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols + ["surveyor_note"]),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])),
              ("clf", LogisticRegression(max_iter=1000))]),
    leaked_B, y, cv=cv, scoring="roc_auc")

print(f"clean                        : {clean.mean():.3f}")
print(f"leak A (preprocess before CV): {a.mean():.3f}   ({a.mean() - clean.mean():+.3f})"
      "   <- often small, but you cannot know that in advance")
print(f"leak B (post-outcome column) : {b.mean():.3f}   ({b.mean() - clean.mean():+.3f})")
print("\nLeak B is the one that ends careers: the score is excellent, the pipeline is")
print("textbook-correct, and the feature simply does not exist at prediction time.")

## Exercises

1. **Group leakage.** Add a `building_id` column with 200 buildings across 800 flats, then
   compare `StratifiedKFold` against `GroupKFold` on it. How much of the clean score was
   really memorisation of buildings?
2. **Target encoding.** Replace the one-hot encoder with `TargetEncoder`. Do it once inside
   the pipeline and once fitted on all rows beforehand, and compare.
3. **Run the checklist.** Answer all five questions from the session-9 handout about *your
   project's* data, in writing. Bring the answers to the clinic on 17 November.